In [ ]:
names_latency=['submission_time', 'duration', 'op2', 'write_size', 'op3']
columns = ['submission_time', 'duration', 'write_size']

# Step 1: Read the CSV file into a Pandas DataFrame
df = pd.read_csv(f'/home/surbhi/measurements/worst_case/STL/1M/uniform/25Util/lat_log_avg_lat.1.log', names=names_latency, usecols=columns)
#df = pd.read_csv(f'/home/surbhi/measurements/worst_case/STL/1M/90-10-LBA/90Util/run1/lat_log_avg_lat.1.log', names=names_latency, usecols=columns)
df['submission_time'] = np.floor(df['submission_time'] - df.iloc[0].submission_time)
df['duration'] = df['duration']/1e6 #convert ns to ms
df['completion_time'] = np.floor(df['submission_time'] + df['duration'])



# Step 3: Determine the start and end time of the entire test
start_time = int(df['submission_time'].min())
end_time = int(df['completion_time'].max())


# Step 4: Create new DataFrames to represent each  millisecond between the start and end time of the entire test
data_written_ms = pd.DataFrame(index=range(start_time, end_time + 1, 1), columns=['data_written_bytes'])
data_written_ms['data_written_bytes'] = 0

# Step 5: Iterate through each write request, calculate the data size for each time interval, and update the corresponding entries in the DataFrames
for index, row in df.iterrows():
    duration = row['duration']
    start = int(row['submission_time'])
    end = start+math.floor(duration)
    
    payload = (1024 * 1024)
    write_rate_per_ms = payload / max(duration, 1.)
    assert(write_rate_per_ms <= payload)
    residue = payload - write_rate_per_ms * math.floor(duration)
    assert (payload >= (write_rate_per_ms * math.floor(duration)))
    data_written_ms.loc[start:end-1] += write_rate_per_ms
    # residue could be zero when last == end
    data_written_ms.loc[end] += residue

print(index)
data_written_ms.reset_index(drop=True, inplace=True)

chunk_size = 10000
results = []
for i in range(0, len(data_written_ms), chunk_size):
    chunk = data_written_ms.iloc[i:i+chunk_size]
    result = chunk.groupby(custom_grouping).sum()
    results.append(result)

df_grouped = pd.concat(results)

# Reset the index to make it a regular column
df_grouped.reset_index(inplace=True)
df_grouped['cumulative_gb'] = df_grouped['data_written_bytes'].cumsum() / (1024 ** 3)
df_grouped['data_written_bytes'] = df_grouped['data_written_bytes'] / (1024 ** 2) 

# Filter the DataFrame for rows where 'data_written_bytes' is less than 75
filtered_df = df_grouped[df_grouped['data_written_bytes'] < 20]
# Get the corresponding 'cumulative_gb' values
cumulative_gb_values = filtered_df['cumulative_gb']
# If you want the first occurrence where 'data_written_bytes' is less than 75
first_cumulative_gb = cumulative_gb_values.iloc[0] if not cumulative_gb_values.empty else 45


#
#data_array = df_grouped['data_written_bytes'].values.reshape(-1, 1)
#print(df_grouped.describe())
#print(df_grouped.info())
print("Min Idx: \n" + str(df_grouped['data_written_bytes'].idxmin()))

data_array = df_grouped['data_written_bytes'].values.reshape(-1, 1)
kmeans = KMeans(n_clusters=2)
kmeans.fit(data_array)
# Get the cluster centers
cluster_centers = kmeans.cluster_centers_.flatten()
# Sort cluster centers to get the lowest and highest modes
cluster_centers.sort()
# Calculate the average of each mode
average_mode1 = cluster_centers[0]
average_mode2 = cluster_centers[1]
print("Average1: " + str(round(average_mode1, )))
print("Average2: " + str(round(average_mode2, )))
avg_bw_str1 = str(round(average_mode1, 2)) + 'MB/sec'
avg_bw_str2 = str(round(average_mode2, 2)) + 'MB/sec'
COLORS = {
    'plot1': '#4477AA',  # Blue
    'plot2': '#EE6677',  # Red  
    'plot3': '#228833',  # Green
    'high_bw': '#CCBB44',  # Yellow
    'low_bw': '#AA3377',   # Purple
    'cache': '#999933',    # Olive
    'end_exp': '#BBBBBB'   # Gray
}

# Set up the plotting style
plt.style.use('default')
plt.rcParams.update({
    'font.size': 12,
    'axes.labelsize': 14,
    'axes.titlesize': 16,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 11,
    'figure.figsize': (10, 6),
    'figure.dpi': 100,
    'lines.linewidth': 2,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.linewidth': 1.2,
})

fig, ax = plt.subplots(figsize=(12, 7))

ax.axhline(y=average_mode2, color=COLORS['high_bw'], linestyle='-', label='Avg Higher Bandwidth: ' + avg_bw_str2)
ax.axhline(y=average_mode1, color=COLORS['low_bw'], linestyle='-', label='Avg Lower Bandwidth: ' + avg_bw_str1)
ax.axvline(x=first_cumulative_gb, color=COLORS['cache'], linestyle=':', label="cache size: " + str(round(first_cumulative_gb, 10)) + "GB")
# Plot vertical lines at x where y is higher than the threshold

#cid = plt.figure().canvas.mpl_connect('button_press_event', mouse_event)

ax.legend(loc='center right', frameon=True, fancybox=True, shadow=True)
# Plot the data size over time for each resolution
plt.tight_layout()
ax.set_ylim(0 , 250)
plot_color = "plot1"
ax.plot(df_grouped['cumulative_gb'], df_grouped['data_written_bytes'], color=COLORS[plot_color], linewidth=2.5, alpha=0.8, label='CGBW')
#plt.plot(df_grouped.index, df_grouped['data_written_bytes'])
ax.set_xlabel('Cumulative Data Written (GB)', fontweight='bold')
ax.set_ylabel('Bandwidth (MB/s)', fontweight='bold')
# Grid
ax.grid(True, alpha=0.3, linestyle='-', linewidth=0.5)
ax.set_title('DM-Hybrid Performance: 1MB Writes, 45GB Burst\n Uniform Random, 25% Zone Utilization', 
                 fontweight='bold', pad=20)
    
plt.savefig('/home/surbhi/github/surbhi-plots/new/STL/1M/1M_Uniform_25Util_DMHybrid_CGBW.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()


fig, ax = plt.subplots(figsize=(12, 7))
plot_color = 'plot1'
ax.plot(df_grouped.index, df_grouped['data_written_bytes'], color=COLORS[plot_color], linewidth=2.5, alpha=0.8, label='Bandwidth')
ax.axhline(y=average_mode2, color=COLORS['high_bw'], linestyle='--', linewidth=2, label=f'Higher BW Mode: {avg_bw_str2}')
ax.axhline(y=average_mode1, color=COLORS['low_bw'], linestyle='--', linewidth=2, label=f'Lower BW Mode: {avg_bw_str1}')
plt.legend(loc='center right', fontsize=14)
#plt.plot(df_grouped.index, df_grouped['data_written_bytes'])


# End of experiment line
xmax = df_grouped.index.max()
xmin = round(int(xmax)/60, 2)
ax.axvline(x=xmax, color=COLORS['end_exp'], linestyle=':', linewidth=2, label=f'End of Experiment: {xmin} minutes')
    
    # Log scales
ax.set_xscale('log', base=10)
ax.set_yscale('log', base=2)


ax.set_xlabel('Time (seconds, $log_{10}$ scale)', fontweight='bold')
ax.set_ylabel('Bandwidth (MB/s, $log_2$ scale)', fontweight='bold')
ax.set_title('DM-Hybrid Bandwidth vs Time: 1MB Writes, 45GB Burst\n25% Zone Utilization, Uniform Random', fontweight='bold', pad=20)
ax.legend(loc='upper right', frameon=True, fancybox=True, shadow=True)
ax.grid(True, alpha=0.3, linestyle='-', linewidth=0.5)


common_yticks = [2**i for i in range(-16, 14, 2)]
common_xticks = [10**i for i in range(0, 6)]
ax.set_yticks(common_yticks)
ax.set_xticks(common_xticks)


# Custom formatter to display ticks as powers of 2
def log2_format(y, pos):
    return f'$2^{{{int(np.log2(y))}}}$' if y > 0 else '0'

# Find the minimum Y-value in the dataset
y_min = min(df_grouped['data_written_bytes'])
# Find the closest lower power of 2
#lowest_power_of_2 = 2 ** np.floor(np.log2(y_min))
lowest_power_of_2 = 2 ** -16
# Set Y-axis limits to include the full range of ticks
# Set Y-axis limits

# Custom formatter to display ticks as powers of 10
def log10_format(y, pos):
    return f'$10^{{{int(np.log10(y))}}}$' if y > 0 else '0'


ax.get_yaxis().set_major_formatter(ticker.FuncFormatter(log2_format))
ax.get_xaxis().set_major_formatter(ticker.FuncFormatter(log10_format))
ax.set_ylim(lowest_power_of_2 , 2**14)

ax.get_xaxis().set_major_formatter(ticker.FuncFormatter(log10_format))
ax.set_xlim(10**0, 10**5)
plt.xticks(common_xticks)

plot_color = 'plot2'
ax.plot(df_grouped.index, df_grouped['data_written_bytes'], color=COLORS[plot_color], label="Bandwidth")
ax.set_title('DM-Hybrid Bandwidth vs Time: 1MB Writes, 45GB Burst\n25% Zone Utilization, Uniform Random', fontweight='bold', pad=20)
plt.savefig('/home/surbhi/github/surbhi-plots/new/STL/1M_25Util_Uniform_DMHybrid_BWTime.pdf', bbox_inches='tight', format="pdf")
plt.show()
print(str(df_grouped.index[-1]))